#  Détection de Bâtiments — Prédiction sur GeoTIFF Drone (Kinshasa)
**Utilise le modèle entraîné par `train_colab.ipynb`**

> **Version géomatique** — lecture GeoTIFF avec `rasterio`, export masques géoréférencés (GeoTIFF) et vectorisation des bâtiments (GeoPackage / Shapefile).

### Structure attendue dans Google Drive :
```
Mon Drive/
├── output/
│   └── best_model.pth              ← produit par train_colab.ipynb
├── Data/drone_images/
│   ├── OAM_Kinshasa_img0.tif
│   ├── OAM_Kinshasa_img1.tif
│   └── OAM_Kinshasa_img2.tif
└── predictions/                    ← créé automatiquement
    ├── *_mask.tif   ← masque GeoTIFF géoréférencé
    ├── *_proba.tif  ← carte de probabilité GeoTIFF
    ├── *_result.png ← visualisation cartographique
    └── batiments_kinshasa.gpkg  ← polygones bâtiments
```

##  Étape 1 — GPU + Installation des dépendances géospatiales

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositif : {device}')
if device.type != 'cuda':
    print('  Aucun GPU — Exécution → Modifier le type d\'exécution → T4 GPU')

!pip install -q albumentations opencv-python-headless tqdm
!pip install -q rasterio geopandas shapely fiona pyproj
print(' Dépendances installées (DL + géospatiales)')

##  Étape 2 — Montage Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print(' Google Drive monté')

##  Étape 3 — Configuration des chemins et vérification des métadonnées spatiales

> Modifiez `DRONE_IMAGES` ou `DRONE_FOLDER` si vos fichiers ont un chemin différent.

In [ ]:
from pathlib import Path
import rasterio

# Chemins
MODEL_PATH   = Path('/content/drive/MyDrive/output/best_model.pth')
PRED_DIR     = Path('/content/drive/MyDrive/predictions')
PRED_DIR.mkdir(parents=True, exist_ok=True)

DRONE_FOLDER = Path('/content/drive/MyDrive/Data/drone_images')
DRONE_IMAGES = [
    DRONE_FOLDER / 'OAM_Kinshasa_img0.tif',
    DRONE_FOLDER / 'OAM_Kinshasa_img1.tif',
    DRONE_FOLDER / 'OAM_Kinshasa_img2.tif',
]

# Paramètres d'inférence
TILE_SIZE    = 512   # identique à l'entraînement
OVERLAP      = 64    # chevauchement pour éviter les artefacts aux bords
THRESHOLD    = 0.5   # seuil de binarisation
MIN_AREA_PX  = 100   # surface minimale d'un bâtiment en pixels²
MIN_AREA_M2  = 9.0   # surface minimale en m² pour l'export vecteur (≈ 3×3 m)
EXPORT_GPKG  = True  # True → GeoPackage, False → Shapefile

# Vérification + affichage des métadonnées spatiales
print(f'Modèle    : {"" if MODEL_PATH.exists() else " INTROUVABLE"} {MODEL_PATH}')
print()
for p in DRONE_IMAGES:
    ok = p.exists()
    print(f'Image : {"" if ok else " INTROUVABLE"} {p.name}')
    if ok:
        with rasterio.open(str(p)) as src:
            print(f'  CRS         : {src.crs}')
            print(f'  Résolution  : {src.res[0]:.6f} × {src.res[1]:.6f}  (unités CRS)')
            print(f'  Dimensions  : {src.width} × {src.height} px  |  {src.count} bande(s)')
            print(f'  Type pixel  : {src.dtypes[0]}')
            print(f'  Emprise     : {src.bounds}')
    print()
print(f'Prédictions → {PRED_DIR}')

##  Étape 4 — Définition du modèle U-Net (identique à l'entraînement)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=[64,128,256,512]):
        super().__init__()
        self.downs = nn.ModuleList(); self.ups = nn.ModuleList(); self.pool = nn.MaxPool2d(2, 2)
        ch = in_channels
        for f in features: self.downs.append(DoubleConv(ch, f)); ch = f
        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        for f in reversed(features):
            self.ups.append(nn.ConvTranspose2d(f*2, f, kernel_size=2, stride=2))
            self.ups.append(DoubleConv(f*2, f))
        self.final_conv = nn.Conv2d(features[0], out_channels, 1)

    def forward(self, x):
        skips = []
        for down in self.downs: x = down(x); skips.append(x); x = self.pool(x)
        x = self.bottleneck(x); skips = skips[::-1]
        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x); skip = skips[i//2]
            if x.shape != skip.shape: x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
            x = torch.cat([skip, x], dim=1); x = self.ups[i+1](x)
        return self.final_conv(x)

ckpt  = torch.load(str(MODEL_PATH), map_location=device)
model = UNet().to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f' Modèle chargé (époque {ckpt["epoch"]} | Dice={ckpt["val_dice"]:.4f} | IoU={ckpt["val_iou"]:.4f})')

##  Étape 5 — Fonctions d'inférence et post-traitement

In [ ]:
import numpy as np
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2

NORM = A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225))
TO_T = ToTensorV2()

def preprocess(tile):
    """(H,W,3) uint8 → tenseur normalisé (1,3,H,W)"""
    t = TO_T(image=NORM(image=tile)['image'])['image']
    return t.unsqueeze(0)

def predict_full_image(model, image_np, tile_size=512, overlap=64, threshold=0.5):
    """Inférence par tuiles avec chevauchement sur une image de taille quelconque."""
    H, W, _ = image_np.shape
    step     = tile_size - overlap
    prob_map = np.zeros((H, W), dtype=np.float32)
    count    = np.zeros((H, W), dtype=np.float32)
    with torch.no_grad():
        for y in range(0, H, step):
            for x in range(0, W, step):
                y2 = min(y + tile_size, H); x2 = min(x + tile_size, W)
                y1 = max(y2 - tile_size, 0); x1 = max(x2 - tile_size, 0)
                tile = image_np[y1:y2, x1:x2]
                ph = tile_size - tile.shape[0]; pw = tile_size - tile.shape[1]
                if ph > 0 or pw > 0: tile = np.pad(tile, ((0,ph),(0,pw),(0,0)), mode='reflect')
                prob = torch.sigmoid(model(preprocess(tile).to(device)))[0,0].cpu().numpy()
                prob = prob[:y2-y1, :x2-x1]
                prob_map[y1:y2, x1:x2] += prob; count[y1:y2, x1:x2] += 1.0
    prob_map /= np.maximum(count, 1)
    return (prob_map > threshold).astype(np.uint8) * 255, prob_map

def post_process(mask, min_area=100):
    """Fermeture morphologique + suppression des petits composants."""
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5,5))
    closed = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(closed, connectivity=8)
    clean = np.zeros_like(closed)
    for i in range(1, n_labels):
        if stats[i, cv2.CC_STAT_AREA] >= min_area: clean[labels == i] = 255
    return clean

def count_buildings(mask):
    _, n = cv2.connectedComponents(mask, connectivity=8)
    return n - 1

print(' Fonctions d\'inférence définies')

##  Étape 6 — Fonctions géospatiales (lecture GeoTIFF, export raster + vecteur)

In [ ]:
import rasterio
from rasterio.crs import CRS
from rasterio.features import shapes
import geopandas as gpd
from shapely.geometry import shape

def read_geotiff(path):
    """Lit un GeoTIFF et retourne (image_np, profile, meta). Gère uint8/uint16/float32, 1-4 bandes."""
    with rasterio.open(str(path)) as src:
        profile = src.profile.copy()
        n_bands = min(src.count, 3)
        bands   = [src.read(i + 1) for i in range(n_bands)]
        if n_bands == 1: bands = bands * 3
        image_np = np.stack(bands, axis=-1)
        if image_np.dtype != np.uint8:
            vmin, vmax = np.nanpercentile(image_np, 2), np.nanpercentile(image_np, 98)
            if vmax > vmin: image_np = np.clip((image_np - vmin) / (vmax - vmin) * 255, 0, 255).astype(np.uint8)
            else: image_np = np.zeros_like(image_np, dtype=np.uint8)
        meta = {'crs': src.crs, 'transform': src.transform, 'width': src.width,
                'height': src.height, 'res_x': src.res[0], 'res_y': src.res[1], 'bounds': src.bounds}
    return image_np, profile, meta

def save_mask_geotiff(mask_uint8, profile, out_path):
    """Sauvegarde le masque binaire (0/255) en GeoTIFF géoréférencé."""
    prof = profile.copy()
    prof.update({'count': 1, 'dtype': 'uint8', 'compress': 'lzw', 'driver': 'GTiff', 'nodata': None})
    with rasterio.open(str(out_path), 'w', **prof) as dst: dst.write(mask_uint8, 1)
    print(f'   Masque GeoTIFF → {out_path.name}')

def save_proba_geotiff(prob_map, profile, out_path):
    """Sauvegarde la carte de probabilité [0,1] en GeoTIFF float32 géoréférencé."""
    prof = profile.copy()
    prof.update({'count': 1, 'dtype': 'float32', 'compress': 'lzw', 'driver': 'GTiff', 'nodata': -9999.0})
    with rasterio.open(str(out_path), 'w', **prof) as dst: dst.write(prob_map.astype('float32'), 1)
    print(f'   Probabilité GeoTIFF → {out_path.name}')

def mask_to_polygons(mask_uint8, transform, crs, min_area_m2=9.0, source_name=''):
    """Vectorise le masque binaire → GeoDataFrame polygones."""
    binary = (mask_uint8 > 0).astype(np.uint8)
    geoms  = [shape(geom) for geom, val in shapes(binary, mask=binary, transform=transform) if val == 1]
    if not geoms:
        print('    Aucun polygone détecté.')
        return gpd.GeoDataFrame(columns=['bat_id','source','area_m2','geometry'], crs=crs)
    gdf = gpd.GeoDataFrame({'geometry': geoms}, crs=crs)
    try:
        gdf_m = gdf.to_crs(epsg=3857); gdf['area_m2'] = gdf_m.area.values
        gdf = gdf[gdf['area_m2'] >= min_area_m2].copy()
    except Exception as e:
        print(f'    Calcul de surface échoué ({e}) — filtre surface non appliqué.')
        gdf['area_m2'] = 0.0
    gdf = gdf.reset_index(drop=True)
    gdf.insert(0, 'bat_id', gdf.index + 1); gdf.insert(1, 'source', source_name)
    return gdf

print(' Fonctions géospatiales définies')

##  Étape 7 — Prédiction sur les 3 images GeoTIFF

In [ ]:
results = []; gdfs = []

for img_path in DRONE_IMAGES:
    print(f'\n{'─'*45}')
    print(f'→ Traitement : {img_path.name}')
    print(f'{'─'*45}')
    if not img_path.exists(): print(f'   Fichier introuvable — ignoré.'); continue

    image_np, profile, meta = read_geotiff(img_path)
    print(f'  Taille   : {meta["width"]} × {meta["height"]} px')
    print(f'  CRS      : {meta["crs"]}')
    print(f'  Résol.   : {meta["res_x"]:.6f} unités/px')

    mask_raw, prob_map   = predict_full_image(model, image_np, TILE_SIZE, OVERLAP, THRESHOLD)
    mask_clean           = post_process(mask_raw, MIN_AREA_PX)
    nb_buildings         = count_buildings(mask_clean)
    print(f'  Bâtiments détectés : {nb_buildings}')

    stem = img_path.stem
    save_mask_geotiff(mask_clean, profile, PRED_DIR / f'{stem}_mask.tif')
    save_proba_geotiff(prob_map,  profile, PRED_DIR / f'{stem}_proba.tif')

    gdf = mask_to_polygons(mask_clean, meta['transform'], meta['crs'], MIN_AREA_M2, img_path.name)
    print(f'  Polygones vectorisés : {len(gdf)}')
    gdfs.append(gdf)
    results.append({'name': img_path.name, 'stem': stem, 'image_np': image_np,
                    'prob_map': prob_map, 'mask_clean': mask_clean,
                    'buildings': nb_buildings, 'meta': meta, 'gdf': gdf})

print('\n Prédictions terminées')

##  Étape 8 — Export vectoriel global (GeoPackage)

In [ ]:
import warnings, pandas as pd
warnings.filterwarnings('ignore')

if gdfs and EXPORT_GPKG:
    gdfs_wgs84 = []
    for gdf in gdfs:
        try: gdfs_wgs84.append(gdf.to_crs(epsg=4326))
        except: gdfs_wgs84.append(gdf)

    all_buildings = pd.concat(gdfs_wgs84, ignore_index=True)
    all_buildings['bat_id'] = range(1, len(all_buildings) + 1)
    gpkg_path = PRED_DIR / 'batiments_kinshasa.gpkg'
    all_buildings.to_file(str(gpkg_path), layer='batiments_wgs84', driver='GPKG')

    native_crss = [gdf.crs for gdf in gdfs if not gdf.empty]
    if native_crss and all(str(c) == str(native_crss[0]) for c in native_crss):
        all_native = pd.concat(gdfs, ignore_index=True)
        all_native['bat_id'] = range(1, len(all_native) + 1)
        all_native.to_file(str(gpkg_path), layer='batiments_natif', driver='GPKG')
        print(f'  Couche CRS natif ({native_crss[0]}) ajoutée.')

    print(f'\n GeoPackage exporté : {gpkg_path}')
    print(f'   Total bâtiments : {len(all_buildings)}')
    print(f'   Colonnes        : {list(all_buildings.columns)}')
    print(f'   CRS (WGS84)     : EPSG:4326')

##  Étape 9 — Visualisation cartographique et sauvegarde

In [ ]:
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
from mpl_toolkits.axes_grid1 import make_axes_locatable

def overlay_mask(image_np, mask, alpha=0.45, color=(255, 0, 0)):
    out = image_np.copy().astype(np.float32); m = mask > 0
    for c, col in enumerate(color): out[..., c][m] = out[..., c][m] * (1 - alpha) + col * alpha
    return out.clip(0, 255).astype(np.uint8)

for r in results:
    meta = r['meta']; b = meta['bounds']
    ext  = [b.left, b.right, b.bottom, b.top]

    fig, axes = plt.subplots(2, 2, figsize=(16, 13))
    fig.suptitle(f"{r['name']} — {r['buildings']} bâtiment(s) | CRS: {meta['crs']}",
                 fontsize=12, fontweight='bold')

    axes[0,0].imshow(r['image_np'], extent=ext, aspect='auto')
    axes[0,0].set_title('Image drone (RGB)'); axes[0,0].set_xlabel('X (CRS)'); axes[0,0].set_ylabel('Y (CRS)')

    im = axes[0,1].imshow(r['prob_map'], cmap='hot', vmin=0, vmax=1, extent=ext, aspect='auto', origin='upper')
    axes[0,1].set_title('Carte de probabilité [0,1]'); axes[0,1].set_xlabel('X (CRS)')
    divider = make_axes_locatable(axes[0,1])
    plt.colorbar(im, cax=divider.append_axes('right', size='4%', pad=0.05))

    axes[1,0].imshow(r['mask_clean'], cmap='gray', extent=ext, aspect='auto', origin='upper')
    axes[1,0].set_title('Masque binaire (post-traitement)'); axes[1,0].set_xlabel('X (CRS)'); axes[1,0].set_ylabel('Y (CRS)')

    overlaid = overlay_mask(r['image_np'], r['mask_clean'])
    axes[1,1].imshow(overlaid, extent=ext, aspect='auto')
    if not r['gdf'].empty:
        gdf_plot = r['gdf'].to_crs(meta['crs']) if r['gdf'].crs != meta['crs'] else r['gdf']
        gdf_plot.boundary.plot(ax=axes[1,1], color='cyan', linewidth=0.6)
    axes[1,1].set_title('Superposition (rouge=masque, cyan=polygones)'); axes[1,1].set_xlabel('X (CRS)')
    patch_r = mpatches.Patch(color='red',  alpha=0.6, label='Masque bâtiment')
    patch_c = mpatches.Patch(edgecolor='cyan', facecolor='none', label='Polygone vecteur')
    axes[1,1].legend(handles=[patch_r, patch_c], loc='lower right', fontsize=8)

    plt.tight_layout()
    out_png = str(PRED_DIR / f"{r['stem']}_result.png")
    plt.savefig(out_png, dpi=150, bbox_inches='tight'); plt.show()
    print(f'   Figure → {out_png}')

##  Étape 10 — Tableau récapitulatif

In [ ]:
print('\n' + '='*70)
print('  RÉCAPITULATIF DES DÉTECTIONS — KINSHASA')
print('='*70)
print(f'  {"Image":<35}  {"Bâtiments (px)":>14}  {"Polygones (vect.)":>17}')
print('-'*70)
for r in results:
    nb_vect = len(r['gdf']) if r['gdf'] is not None else 0
    print(f'  {r["name"]:<35}  {r["buildings"]:>14}  {nb_vect:>17}')
print('='*70)
print(f'\n  Résultats dans : {PRED_DIR}')
print('  Fichiers produits :')
print('    *.tif        → masques et probabilités géoréférencés (ouvrir dans QGIS)')
print('    *.gpkg       → polygones bâtiments (GeoPackage, couches WGS84 + natif)')
print('    *_result.png → visualisations cartographiques')